# Modern Training Best Practices

This notebook consolidates the techniques used in state-of-the-art deep learning pipelines into one practical reference.

**Topics covered:**
- Gradient accumulation — simulate large batches on small GPUs
- Learning rate warmup + cosine annealing
- Model checkpointing — save and restore the best model
- Early stopping — prevent overfitting automatically
- Weight decay vs Dropout — when to use which
- A complete production-ready training loop combining all techniques


## 1. Gradient Accumulation

**Problem:** Large batch sizes improve training stability, but a batch of 512 images may not fit in GPU memory.

**Solution:** Split the batch into smaller micro-batches, accumulate gradients over N steps, then update:

```
for step, (imgs, labels) in enumerate(loader):
    loss = criterion(model(imgs), labels) / accumulation_steps
    loss.backward()                        # accumulate
    if (step + 1) % accumulation_steps == 0:
        optimizer.step()                   # update once
        optimizer.zero_grad()
```

**Effect:** `effective_batch_size = batch_size × accumulation_steps`


In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dataset
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
train_ds = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=tf)
val_ds   = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=tf)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(128*8*8,256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256,10)
        )
    def forward(self, x): return self.net(x)

# Gradient accumulation demo
ACCUM_STEPS = 4   # effective batch = 64 × 4 = 256
model = SimpleCNN().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

model.train()
optimizer.zero_grad()
for step, (imgs, labels) in enumerate(train_loader):
    imgs, labels = imgs.to(device), labels.to(device)
    loss = criterion(model(imgs), labels) / ACCUM_STEPS   # normalize
    loss.backward()
    if (step + 1) % ACCUM_STEPS == 0:
        optimizer.step()
        optimizer.zero_grad()
    if step >= 19: break   # demo: 20 steps only

print(f"Gradient accumulation demo complete.")
print(f"  Micro-batch size : 64")
print(f"  Accumulation steps: {ACCUM_STEPS}")
print(f"  Effective batch  : {64 * ACCUM_STEPS}")


## 2. Learning Rate Warmup + Cosine Annealing

**Warmup:** Start with a very small LR, ramp it up linearly over the first few epochs.  
This prevents the untrained model from making destructive early updates.

**Cosine Annealing:** After warmup, decay LR smoothly following a cosine curve.  
This combination is used in ResNets, ViT, BERT, and virtually all modern models.

```
Epoch:  0  1  2  3  4 ... 30
LR:    ↑  ↑  ↑ peak ↘  ↘ ... ≈0
         warmup    cosine decay
```


In [ ]:
import torch.optim.lr_scheduler as sched
import numpy as np

# Warmup + Cosine annealing
def get_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / max(1, warmup_epochs)          # linear warmup
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))      # cosine decay
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Visualize
dummy_opt = torch.optim.SGD([torch.tensor(0.)], lr=0.1)
scheduler = get_warmup_cosine_scheduler(dummy_opt, warmup_epochs=5, total_epochs=50)

lrs = []
for _ in range(50):
    lrs.append(dummy_opt.param_groups[0]['lr'])
    scheduler.step()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lrs, color='steelblue', linewidth=2.5)
ax.axvspan(0, 5, alpha=0.15, color='orange', label='Warmup (5 epochs)')
ax.axvspan(5, 50, alpha=0.08, color='blue',   label='Cosine decay')
ax.set(title='Warmup + Cosine Annealing Schedule', xlabel='Epoch', ylabel='LR Multiplier')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 3. Model Checkpointing

Always save the **best** model (by validation accuracy), not just the last one.  
Save the full training state so you can resume if training is interrupted.


In [ ]:
import os

class Checkpointer:
    def __init__(self, path='best_model.pt', mode='max'):
        self.path = path
        self.mode = mode
        self.best = float('-inf') if mode == 'max' else float('inf')

    def save(self, model, optimizer, epoch, metric, scheduler=None):
        improved = (self.mode == 'max' and metric > self.best) or                    (self.mode == 'min' and metric < self.best)
        if improved:
            self.best = metric
            torch.save({
                'epoch':      epoch,
                'model':      model.state_dict(),
                'optimizer':  optimizer.state_dict(),
                'scheduler':  scheduler.state_dict() if scheduler else None,
                'best_metric': self.best,
            }, self.path)
            print(f"  Checkpoint saved  (epoch {epoch}, val_acc={metric:.4f})")
            return True
        return False

    def load(self, model, optimizer=None, scheduler=None):
        ckpt = torch.load(self.path, map_location='cpu')
        model.load_state_dict(ckpt['model'])
        if optimizer and ckpt.get('optimizer'): optimizer.load_state_dict(ckpt['optimizer'])
        if scheduler and ckpt.get('scheduler'): scheduler.load_state_dict(ckpt['scheduler'])
        print(f"  Loaded checkpoint from epoch {ckpt['epoch']} (best={ckpt['best_metric']:.4f})")
        return ckpt['epoch']

print("Checkpointer class defined. Usage:")
print("  ckpt = Checkpointer('best.pt', mode='max')")
print("  ckpt.save(model, optimizer, epoch, val_acc)")
print("  ckpt.load(model, optimizer)   # resume training")


## 4. Early Stopping

Stop training when validation metric stops improving — prevents overfitting and wasted compute.

**Patience:** how many epochs to wait for improvement before stopping.  
**Min delta:** smallest change that counts as improvement.


In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0.001, mode='max'):
        self.patience  = patience
        self.min_delta = min_delta
        self.mode      = mode
        self.best      = None
        self.counter   = 0
        self.stop      = False

    def step(self, metric):
        if self.best is None:
            self.best = metric
            return False
        improved = (self.mode == 'max' and metric > self.best + self.min_delta) or                    (self.mode == 'min' and metric < self.best - self.min_delta)
        if improved:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"  EarlyStopping: no improvement for {self.counter}/{self.patience} epochs")
            if self.counter >= self.patience:
                self.stop = True
        return self.stop

print("EarlyStopping defined. Usage:")
print("  es = EarlyStopping(patience=5, mode='max')")
print("  for epoch in range(100):")
print("      val_acc = evaluate(model, val_loader)")
print("      if es.step(val_acc): break   # stop training!")


## 5. Full Modern Training Loop

Putting it all together: gradient accumulation, warmup+cosine LR, checkpointing, early stopping, and AMP.


In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total

def modern_train(epochs=20, accum_steps=4, warmup=3, patience=5):
    torch.manual_seed(42)
    model     = SimpleCNN().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = get_warmup_cosine_scheduler(optimizer, warmup, epochs)
    scaler    = torch.cuda.amp.GradScaler(enabled=device.type=='cuda')
    ckpt      = Checkpointer('best_modern.pt', mode='max')
    es        = EarlyStopping(patience=patience, mode='max')
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss':[], 'val_acc':[], 'lr':[]}
    for epoch in range(1, epochs+1):
        model.train(); optimizer.zero_grad(); running_loss = 0
        for step, (imgs, labels) in enumerate(train_loader):
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type=='cuda'):
                loss = criterion(model(imgs), labels) / accum_steps
            scaler.scale(loss).backward()
            if (step+1) % accum_steps == 0 or step == len(train_loader)-1:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            running_loss += loss.item() * accum_steps

        val_acc = evaluate(model, val_loader)
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(running_loss/len(train_loader))
        history['val_acc'].append(val_acc); history['lr'].append(lr)
        print(f"Epoch {epoch:2d} | loss={running_loss/len(train_loader):.4f} | val_acc={val_acc:.4f} | lr={lr:.6f}")
        ckpt.save(model, optimizer, epoch, val_acc, scheduler)
        if es.step(val_acc): print("Early stopping triggered!"); break

    return history

history = modern_train(epochs=20)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss']); axes[0].set(title='Training Loss', xlabel='Epoch')
axes[1].plot(history['val_acc']);    axes[1].set(title='Validation Accuracy', xlabel='Epoch')
axes[2].plot(history['lr']);         axes[2].set(title='Learning Rate', xlabel='Epoch')
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 6. Exercises

1. **Patience tuning**: Run `modern_train` with `patience=3` and `patience=10`. How does early stopping affect the final accuracy?
2. **Accumulation comparison**: Compare training with `accum_steps=1` vs `accum_steps=8`. Does the effective batch size affect convergence?
3. **AdamW vs Adam**: Replace `AdamW` with `Adam` (no `weight_decay`). Does removing explicit weight decay hurt validation accuracy?
